# LeWM Push-T: Prescribed vs Free Space

Prescribed axes `(block_x, block_y, block_angle)` vs free (learned) embedding.

**Runtime → Change runtime type → T4 GPU**

Потом жми **Runtime → Run all**. ~10 минут.

In [ ]:
!pip install gym-pusht -q

In [ ]:
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Сбор данных через gym-pusht
State = `[agent_x, agent_y, block_x, block_y, block_angle]`

In [ ]:
import gymnasium as gym
import gym_pusht

def collect_data(n_ep=200, max_steps=300, frameskip=5, seed=42):
    rng = np.random.default_rng(seed)
    episodes = []
    env = gym.make('gym_pusht/PushT-v0', obs_type='state', render_mode=None)

    for i in range(n_ep):
        obs, _ = env.reset(seed=int(rng.integers(0, 100000)))
        states, actions = [obs.copy()], []

        for step in range(max_steps):
            if step == 0 or rng.random() < 0.3:
                act = env.action_space.sample()
            else:
                act = actions[-1] + rng.normal(0, 30, size=2).astype(np.float32)
                act = np.clip(act, 0, 512)

            obs, _, done, trunc, _ = env.step(act)

            if (step + 1) % frameskip == 0:
                states.append(obs.copy())
                actions.append(act.copy())

            if done or trunc:
                break

        if len(actions) >= 4:
            episodes.append((
                np.array(states[:len(actions)+1]),
                np.array(actions)
            ))

        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{n_ep} episodes')

    env.close()
    lens = [len(e[1]) for e in episodes]
    print(f'Done: {len(episodes)} episodes, avg len {np.mean(lens):.0f}')
    return episodes

episodes = collect_data(200)

In [ ]:
# Проверяем данные
s0 = episodes[0][0][0]
print(f'State example: {s0}')
print(f'  agent_x={s0[0]:.1f}, agent_y={s0[1]:.1f}')
print(f'  block_x={s0[2]:.1f}, block_y={s0[3]:.1f}, block_angle={s0[4]:.3f}')
print(f'Action example: {episodes[0][1][0]}')

## 2. Dataset + Models

In [ ]:
H = 3  # history size (как в LeWM)

class SeqDataset(Dataset):
    def __init__(self, episodes, H=3):
        self.windows = []
        for states, actions in episodes:
            for t in range(len(actions) - H):
                self.windows.append((
                    states[t:t+H+2].astype(np.float32),  # H+2 states
                    actions[t:t+H+1].astype(np.float32)  # H+1 actions
                ))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        s, a = self.windows[idx]
        return torch.from_numpy(s), torch.from_numpy(a)

ds = SeqDataset(episodes, H)
print(f'Total windows: {len(ds)}')

In [ ]:
# SIGReg из LeWM
class SIGReg(nn.Module):
    def __init__(self, knots=17, num_proj=512):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots)
        dt = 3 / (knots - 1)
        w = torch.full((knots,), 2*dt); w[[0,-1]] = dt
        phi = torch.exp(-t.square() / 2.0)
        self.register_buffer('t', t)
        self.register_buffer('phi', phi)
        self.register_buffer('weights', w * phi)

    def forward(self, proj):
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        return ((err @ self.weights) * proj.size(-2)).mean()

## 3. Эксперимент

In [ ]:
def run_experiment(mode, episodes, seed=42, D=3, H=3, epochs=50,
                   bs=256, lr=3e-4, lam=0.09, use_sigreg=True):
    """
    mode: 'prescribed_block', 'prescribed_full', 'free'
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds = SeqDataset(episodes, H)
    nt = int(len(ds) * 0.9); nv = len(ds) - nt
    tr, va = random_split(ds, [nt, nv],
                          generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(tr, batch_size=bs, shuffle=True, drop_last=True)
    vl = DataLoader(va, batch_size=bs)

    # --- Encoder ---
    scale3 = torch.tensor([1/512, 1/512, 1/(2*np.pi)], device=device)
    scale5 = torch.tensor([1/512, 1/512, 1/512, 1/512, 1/(2*np.pi)], device=device)

    if mode == 'prescribed_block':
        enc_dim = 3
        enc = lambda s: s[..., 2:5] * scale3
        enc_params = []
    elif mode == 'prescribed_full':
        enc_dim = 5
        enc = lambda s: s * scale5
        enc_params = []
    else:  # free
        enc_dim = D
        enc_net = nn.Sequential(
            nn.Linear(5, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, D)
        ).to(device)
        enc = enc_net
        enc_params = list(enc_net.parameters())

    # --- Action encoder + Predictor ---
    aenc = nn.Sequential(
        nn.Linear(2, 32), nn.GELU(), nn.Linear(32, enc_dim)
    ).to(device)

    pred = nn.Sequential(
        nn.Linear(H * 2 * enc_dim, 128), nn.LayerNorm(128), nn.GELU(),
        nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(),
        nn.Linear(128, enc_dim)
    ).to(device)

    sigreg = SIGReg(17, 512).to(device) if use_sigreg else None

    params = enc_params + list(aenc.parameters()) + list(pred.parameters())
    n_params = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

    print(f'\n--- {mode} (params={n_params:,}, D={enc_dim}, sigreg={use_sigreg}) ---')

    history = []
    best_vp, best_ep = float('inf'), 0

    for ep in range(1, epochs + 1):
        t0 = time.time()

        # Train
        if hasattr(enc, 'train'): enc.train()
        aenc.train(); pred.train()
        train_pl, train_sl, n = 0, 0, 0

        for s, a in tl:
            s, a = s.to(device), a.to(device)
            emb = enc(s)
            ctx, tgt = emb[:, :H], emb[:, H]
            ae = aenc(a[:, :H])
            p = pred(torch.cat([ctx, ae], dim=-1).reshape(s.size(0), -1))

            pl = F.mse_loss(p, tgt.detach())
            sl = sigreg(emb.transpose(0, 1)) if sigreg else torch.tensor(0.0)
            loss = pl + lam * sl

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()

            b = s.size(0)
            train_pl += pl.item() * b
            train_sl += sl.item() * b
            n += b

        train_pl /= n
        train_sl /= n

        # Val
        if hasattr(enc, 'eval'): enc.eval()
        aenc.eval(); pred.eval()
        val_pl, val_n = 0, 0
        all_pred, all_tgt = [], []

        with torch.no_grad():
            for s, a in vl:
                s, a = s.to(device), a.to(device)
                emb = enc(s)
                ctx, tgt = emb[:, :H], emb[:, H]
                ae = aenc(a[:, :H])
                p = pred(torch.cat([ctx, ae], dim=-1).reshape(s.size(0), -1))

                val_pl += F.mse_loss(p, tgt).item() * s.size(0)
                val_n += s.size(0)
                all_pred.append(p.cpu())
                all_tgt.append(tgt.cpu())

        vp = val_pl / val_n
        pp, tt = torch.cat(all_pred), torch.cat(all_tgt)
        per_axis = (pp - tt).pow(2).mean(0).tolist()

        if vp < best_vp:
            best_vp, best_ep = vp, ep

        sch.step()
        elapsed = time.time() - t0
        history.append({'ep': ep, 'tp': train_pl, 'vp': vp, 'axes': per_axis})

        if ep % 10 == 0 or ep == 1:
            ax = ', '.join(f'{v:.6f}' for v in per_axis)
            print(f'  ep {ep:3d} | train {train_pl:.6f} | '
                  f'val {vp:.6f} | sig {train_sl:.4f} | '
                  f'[{ax}] | {elapsed:.1f}s')

    print(f'  BEST: {best_vp:.6f} (ep {best_ep})')
    return {
        'mode': mode, 'params': n_params, 'best_vp': best_vp,
        'best_ep': best_ep, 'history': history
    }

In [ ]:
# === MAIN RUN ===
results = {}
for seed in [42, 123, 777]:
    print(f'\n{"="*60}')
    print(f'  SEED {seed}')
    print(f'{"="*60}')
    for mode in ['prescribed_block', 'prescribed_full', 'free']:
        r = run_experiment(mode, episodes, seed=seed, epochs=50)
        results[f'{seed}_{mode}'] = r

## 4. Результаты

In [ ]:
print(f'\n{"="*70}')
print(f'{"RESULTS":^70}')
print(f'{"="*70}')
print(f'{"seed":>6s} {"presc_block":>14s} {"presc_full":>14s} {"free":>14s}')
print('-' * 50)

for seed in [42, 123, 777]:
    vals = [results[f'{seed}_{m}']['best_vp']
            for m in ['prescribed_block', 'prescribed_full', 'free']]
    best_i = np.argmin(vals)
    row = f'  {seed}  '
    for i, v in enumerate(vals):
        s = f'{v:.6f}'
        row += f'  {"*"+s+"*" if i==best_i else " "+s+" "}'
    print(row)

avgs = {m: np.mean([results[f'{s}_{m}']['best_vp']
                    for s in [42, 123, 777]])
        for m in ['prescribed_block', 'prescribed_full', 'free']}
print('-' * 50)
print(f'  avg  ' + '  '.join(f' {avgs[m]:.6f} '
      for m in ['prescribed_block', 'prescribed_full', 'free']))

ratio = avgs['free'] / avgs['prescribed_block']
print(f'\nFree / Prescribed_block = {ratio:.1f}x')
print(f'Prescribed block params: {results["42_prescribed_block"]["params"]:,}')
print(f'Free params: {results["42_free"]["params"]:,}')

In [ ]:
# === Training curves ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'prescribed_block': '#2ecc71', 'prescribed_full': '#3498db', 'free': '#e74c3c'}

for i, seed in enumerate([42, 123, 777]):
    ax = axes[i]
    for mode in ['prescribed_block', 'prescribed_full', 'free']:
        h = results[f'{seed}_{mode}']['history']
        ax.plot([x['ep'] for x in h], [x['vp'] for x in h],
                label=mode, color=colors[mode], alpha=0.8)
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val pred_loss')
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('LeWM Push-T: Prescribed vs Free', fontsize=14)
plt.tight_layout()
plt.savefig('lewm_pusht_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lewm_pusht_results.png')

In [ ]:
# === Per-axis analysis (last epoch) ===
print('\nPer-axis MSE (last epoch):')
for seed in [42, 123, 777]:
    print(f'\n  Seed {seed}:')
    for mode in ['prescribed_block', 'free']:
        h = results[f'{seed}_{mode}']['history'][-1]
        ax = h['axes']
        if mode == 'prescribed_block':
            names = ['block_x', 'block_y', 'angle']
        else:
            names = [f'dim_{i}' for i in range(len(ax))]
        parts = ', '.join(f'{n}={v:.6f}' for n, v in zip(names, ax))
        print(f'    {mode}: {parts}')

## 5. Без SIGReg (ablation)

In [ ]:
# Проверяем: SIGReg помогает free или нет?
print('Ablation: without SIGReg (seed=42)')
for mode in ['prescribed_block', 'free']:
    r = run_experiment(mode, episodes, seed=42, epochs=50, use_sigreg=False)
    print(f'  {mode} no_sigreg: {r["best_vp"]:.6f}')
    print(f'  {mode} w/ sigreg: {results[f"42_{mode}"]["best_vp"]:.6f}')